# Workshop 3: Sequential Movie Recommendation via Dueling Double Deep Q-Networks (D3QN) on MovieLens 32M

**Course:** Machine Learning (2026-III) — Computer Engineering Program, Universidad Distrital Francisco José de Caldas  
**Professor:** Eng. Carlos Andrés Sierra, M.Sc.  
**Authors (Team 7):** Andrés Felipe Ruiz Vasallo (`afruizv@udistrital.edu.co`), Josep Emanuel León Joya (`jeleonj@udistrital.edu.co`), Matthew Stev Toro Mondragón (`mstorom@udistrital.edu.co`)

---
### Notebook Overview
This notebook provides an interactive walkthrough of **Workshop 3**, formulating sequential movie recommendation as a finite-horizon Markov Decision Process (MDP) $(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ and solving it with a **Dueling Double Deep Q-Network (D3QN)** agent:
1. **MDP Environment (`MovieLensRecEnv`)**: $d_s=62$ continuous state representation combining SVD user taste vectors, exponentially decayed session history ($\beta=0.7$), 20-genre exposure tracking, diversity entropy, and session fatigue counters over $K=20$ candidate slate actions ($H=20$ steps, $\gamma=0.95$).
2. **Dueling Double DQN (`DuelingDQN`)**: Decoupled value $V(s;\theta,\beta)$ and advantage $A(s,a;\theta,\alpha)$ streams with mean-advantage identifiability, Double Q-learning target decoupling, Huber (Smooth L1) TD loss, and $\varepsilon$-greedy / Softmax exploration.
3. **Non-RL & Tabular RL Baselines**: Random, Global Popularity, Static SVD Greedy (Workshop 1 baseline), Myopic LinUCB Contextual Bandit ($\gamma=0$), and Discretized Tabular Q-Learning.
4. **Empirical Results & Diagnostic Visualizations**: Learning curves, exploration schedules, test benchmark tables with 95% confidence intervals, sample trajectories, and sample efficiency.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display

# Resolve Workshop 3 root directory
NOTEBOOK_DIR = Path.cwd()
W3_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else Path("workshop3").resolve()
if str(W3_DIR.parent) not in sys.path:
    sys.path.insert(0, str(W3_DIR.parent))
if str(W3_DIR / "src") not in sys.path:
    sys.path.insert(0, str(W3_DIR / "src"))

print(f"Workshop 3 Directory: {W3_DIR}")
print(f"PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

## 1. Gymnasium MDP Environment (`MovieLensRecEnv`) & Random Policy Diagnostics

At each step $t \in \{0,\dots,H-1\}$, the agent observes state $s_t \in \mathbb{R}^{62}$ and selects slate index $a_t \in \{0,\dots,19\}$. The shaped multi-objective reward function balances immediate relevance, novelty, intra-session genre diversity, and repetition avoidance:
$$R(s_t, a_t) = r_{\text{rel}}(u, i_{a_t}) + \lambda_{\text{div}} \cdot \text{div}(i_{a_t}, \mathcal{H}_t) + \lambda_{\text{nov}} \cdot \text{nov}(i_{a_t}) - \lambda_{\text{rep}} \cdot \mathbb{I}[\text{genre\_streak}_t \ge 3]$$

In [2]:
from env import MovieLensRecEnv
from model import DuelingDQN

env = MovieLensRecEnv(split="test", seed=42)
obs, info = env.reset(seed=42)
print(f"Observation Space: {env.observation_space.shape} (dtype={obs.dtype})")
print(f"Action Space:      Discrete({env.action_space.n})")

# Instantiate Dueling Double DQN network and inspect stream shapes
net = DuelingDQN(state_dim=62, action_dim=20)
with torch.no_grad():
    s_tensor = torch.as_tensor(obs, dtype=torch.float32).unsqueeze(0)
    q_vals = net(s_tensor)
print(f"Q(s, a) output shape: {q_vals.shape} | Range: [{q_vals.min().item():.3f}, {q_vals.max().item():.3f}]")

## 2. Held-Out Test Benchmark Across All 6 Policies

We load `workshop3/report/metrics_summary.json` containing the 500-episode held-out chronological test evaluation across all 6 policies (with 95% confidence intervals):

In [3]:
metrics_path = W3_DIR / "report" / "metrics_summary.json"
with open(metrics_path, "r", encoding="utf-8") as f:
    w3_summary = json.load(f)

rows = []
for policy_name, m in w3_summary["test_evaluation"].items():
    rows.append({
        "Policy": policy_name,
        "Return (Mean ± CI95)": f"{m['mean_return']:.2f} ± {m['ci95_return']:.2f}",
        "Discounted Return": round(m["mean_discounted_return"], 2),
        "HitRate (r >= 4.0)": f"{m['mean_hit_rate']*100:.2f}%",
        "Mean Rating": round(m["mean_predicted_rating"], 3),
        "Genre Coverage": round(m["mean_genre_coverage"], 3),
        "Novelty (bits)": round(m["mean_novelty_bits"], 2),
        "Avg Length": round(m["mean_episode_length"], 2),
    })

df_w3 = pd.DataFrame(rows)
display(df_w3)

## 3. Diagnostic Figures (300 DPI)

Below we render the empirical diagnostic plots generated by `workshop3/src/evaluate.py`:

In [4]:
fig_dir = W3_DIR / "report" / "figures"
for fig_name in [
    "fig2_random_policy_diagnostics.png",
    "fig3_hyperparameter_learning_curves.png",
    "fig4_exploration_dynamics.png",
    "fig5_test_metrics_comparison.png",
    "fig6_sample_trajectories.png",
    "fig7_sample_efficiency.png",
]:
    fpath = fig_dir / fig_name
    if fpath.exists():
        print(f"=== {fig_name} ===")
        display(Image(filename=str(fpath), width=850))